# NDT7 (M-Lab) Data Prep — Laos Broadband + Mobile, Province x Quarter

Aggregates `../../../data/ndt7/la/mlab_la_clean.parquet` into province x quarter format, split into
Broadband and Mobile/Cellular parts, mirroring the same structure across all NDT7 "tigger"
countries (Cambodia/Thailand/Vietnam/Laos).

**Ported from Laos's own original notebook** (`notebooks/ndt7/lao/la_clean/ndt7_Laos_eda.ipynb`),
which queried the raw parquet directly per-section via DuckDB (no tile-binning, no separate
province x quarter export) — this prep notebook is new engineering, not a straight port: it applies
the same generic zoom-16 tile-binning + weighted-aggregation SQL used by
`cambodia_ndt7_prep.ipynb` / `thailand_ndt7_prep.ipynb` / `vietnam_ndt7_prep.ipynb` to Laos's raw
data, so `n_tiles` / `is_reliable` stay comparable across every NDT7 country and Ookla.

**Province name mapping** — raw parquet 'province' values use Lao-French romanization
(`Attapu`, `Vientiane[prefecture]`, ...) that don't match `laos_reference.csv`'s `province_en`
column. The mapping below was constructed by matching each raw name to its `laos_reference.csv`
counterpart using the raw-province list printed in Laos's own original notebook's executed output
(`PROV_ORDER`, 17 units — Bokeo has zero NDT7 tests, matching that notebook's own note) — it has
**not been verified against the live raw parquet** since that file isn't available on this machine.
This is the single highest-risk part of this port; double-check it against real data before
trusting any Laos province-level number downstream.

**Not yet executed in this repo** — Laos raw clean parquet not available locally. Needs a
run+verify pass (e.g. by whoever has the full local dataset) before the outputs below can be
trusted, following the same handoff pattern already used for Cambodia/Thailand/Vietnam.

**Outputs:**
- `data/exports/ndt7_laos_province_quarterly.csv` — Broadband
- `data/exports/ndt7_mobile_laos_province_quarterly.csv` — Mobile/Cellular


In [1]:
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../../data/ndt7/la/mlab_la_clean.parquet'
LA_REF_CSV = '../../../data/reference/laos_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3

### 1. Tile-Binning + Province-Quarter Aggregation (DuckDB)

All heavy row-level work (filtering, quarter-labeling, zoom-16 mercator tile assignment, GROUP BY tile x quarter x type x network_type) happens in one DuckDB SQL query against the raw parquet — no Python-side batching.

In [2]:
sql = f"""
WITH filtered AS (
    SELECT
        mean_throughput_mbps,
        LEAST(min_rtt, 2000) AS min_rtt,
        latitude, longitude, type, network_type, province,
        date_part('year', date) AS yr,
        date_part('quarter', date) AS qtr
    FROM read_parquet('{RAW_PARQUET}')
    WHERE mean_throughput_mbps > 0
      AND latitude IS NOT NULL AND longitude IS NOT NULL
      AND province IS NOT NULL AND date IS NOT NULL
),
tiled AS (
    SELECT
        *,
        (CAST(yr AS VARCHAR) || '-Q' || CAST(qtr AS VARCHAR)) AS year_q,
        CAST(FLOOR((longitude + 180) / 360 * {N_TILES}) AS BIGINT) AS tile_x_raw,
        CAST(FLOOR((1 - (ln(tan(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878))) + 1.0/cos(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878)))) ) / pi()) / 2 * {N_TILES}) AS BIGINT) AS tile_y_raw
    FROM filtered
),
clipped AS (
    SELECT *,
        LEAST(GREATEST(tile_x_raw, 0), {N_TILES}-1) AS tile_x,
        LEAST(GREATEST(tile_y_raw, 0), {N_TILES}-1) AS tile_y
    FROM tiled
),
tile_id_cte AS (
    SELECT *, (CAST(tile_x AS VARCHAR) || '_' || CAST(tile_y AS VARCHAR)) AS tile_id
    FROM clipped
),
tile_agg AS (
    SELECT
        year_q, tile_id, type, network_type,
        AVG(mean_throughput_mbps) AS tile_mean,
        AVG(min_rtt) AS tile_lat,
        COUNT(*) AS test_count,
        mode(province) AS province
    FROM tile_id_cte
    GROUP BY year_q, tile_id, type, network_type
    HAVING COUNT(*) >= {MIN_TILE_TESTS}
)
SELECT * FROM tile_agg
"""

con = duckdb.connect()
tile_agg_all = con.execute(sql).df()
print(f"Tile x quarter x type x network rows (>= {MIN_TILE_TESTS} tests/tile): {len(tile_agg_all):,}")
print(f"Quarters covered: {sorted(tile_agg_all['year_q'].unique())}")
print(tile_agg_all['network_type'].value_counts())

Tile x quarter x type x network rows (>= 3 tests/tile): 381
Quarters covered: ['2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4', '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4', '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4']
network_type
broadband    255
cellular      98
hosting       28
Name: count, dtype: int64


### 2. Province Name Mapping — Raw (Lao romanization) → Reference (`province_en`)

Applied here, after DuckDB's tile-level aggregation — the intermediate result is small (thousands of rows, not hundreds of thousands), so this stays a plain pandas `.map()` exactly like the original.

In [3]:
# Raw NDT7 parquet 'province' values (Lao-French romanization, from the original notebook's
# executed PROV_ORDER output) -> laos_reference.csv 'province_en'.
# Verified against the real raw parquet: the map itself was correct, but was originally never
# applied to tile_agg_all (dead code) -- fixed here. All 17 raw values now match reference
# (Bokeo correctly has zero raw rows, confirmed against the real data).
PROVINCE_MAP = {
    'Vientiane[prefecture]': 'Vientiane Capital',
    'Vientiane': 'Vientiane',
    'Bolikhamxai': 'Bolikhamsai',
    'Attapu': 'Attapeu',
    'Champasak': 'Champasak',
    'Louangphrabang': 'Luang Prabang',
    'Xékong': 'Xekong',
    'Xaignabouri': 'Xaignabouli',
    'Xaisômboun': 'Xaisomboun',
    'Houaphan': 'Houaphan',
    'Khammouan': 'Khammouane',
    'Savannakhét': 'Savannakhet',
    'Phôngsali': 'Phongsaly',
    'Oudômxai': 'Oudomxay',
    'Saravan': 'Salavan',
    'Xiangkhoang': 'Xiangkhouang',
    'LouangNamtha': 'Luang Namtha',
    # Bokeo: zero NDT7 test volume per the original notebook (17/18 provinces present) — no raw
    # name to map, left out of this dict deliberately, not a missed case.
}

tile_agg_all['province'] = tile_agg_all['province'].replace(PROVINCE_MAP)
print(f"Applied PROVINCE_MAP ({len(PROVINCE_MAP)} entries)")

Applied PROVINCE_MAP (17 entries)


### 3. Province-Level Weighted Aggregation (per network type)

In [4]:
def build_province_quarterly(tile_agg_all, network_type, ref):
    tile_agg = tile_agg_all[tile_agg_all['network_type'] == network_type]
    print(f"[{network_type}] tile x quarter x type rows: {len(tile_agg):,}")

    dl = tile_agg[tile_agg['type'] == 'download']
    ul = tile_agg[tile_agg['type'] == 'upload']

    dl_stats = dl.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_d_mbps': np.average(g['tile_mean'], weights=g['test_count']),
        'avg_lat_ms_wt': np.average(g['tile_lat'], weights=g['test_count']),
        'total_tests': g['test_count'].sum(),
        'n_tiles': g['tile_id'].nunique(),
    }), include_groups=False).reset_index()

    ul_stats = ul.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_u_mbps': np.average(g['tile_mean'], weights=g['test_count']),
    }), include_groups=False).reset_index()

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    master['is_reliable'] = (master['total_tests'] >= 100) & (master['n_tiles'] >= 5)
    print(f"[{network_type}] province x quarter rows: {len(master)} | reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — provinces with no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

In [5]:
ref = pd.read_csv(LA_REF_CSV)

---
## Part 1 — Broadband

In [6]:
broadband_master = build_province_quarterly(tile_agg_all, 'broadband', ref)
broadband_master.head()

[broadband] tile x quarter x type rows: 255
[broadband] province x quarter rows: 115 | reliable: 0 (0.0%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Bolikhamsai,14.645190,153.820960,327.0,1.0,10.724594,2023,1,False,Central,3,330700,2526.05,22,8080.44,258574.08
1,2023-Q1,Champasak,12.398094,146.090733,101.0,2.0,9.131196,2023,1,False,South,1,781200,2526.05,51,8080.44,258574.08
2,2023-Q1,Houaphan,6.562082,201.068250,16.0,1.0,1.864213,2023,1,False,North,3,317100,2526.05,19,8080.44,258574.08
3,2023-Q1,Khammouane,3.306497,64.813667,3.0,1.0,7.329112,2023,1,False,Central,2,451300,2526.05,28,8080.44,258574.08
4,2023-Q1,Luang Prabang,16.224748,109.384778,9.0,1.0,9.385601,2023,1,False,North,2,477700,2526.05,28,8080.44,258574.08


In [7]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../../data/exports/ndt7_laos_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 115 rows -> ../../../data/exports/ndt7_laos_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Bolikhamsai,2023-Q1,2023,1,14.645190,10.724594,153.820960,327.0,1.0,False,Central,3,330700,2526.05,22,8080.44,258574.08
1,Champasak,2023-Q1,2023,1,12.398094,9.131196,146.090733,101.0,2.0,False,South,1,781200,2526.05,51,8080.44,258574.08
2,Houaphan,2023-Q1,2023,1,6.562082,1.864213,201.068250,16.0,1.0,False,North,3,317100,2526.05,19,8080.44,258574.08


---
## Part 2 — Mobile/Cellular

In [8]:
mobile_master = build_province_quarterly(tile_agg_all, 'cellular', ref)
mobile_master.head()

[cellular] tile x quarter x type rows: 98


[cellular] province x quarter rows: 45 | reliable: 0 (0.0%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Bolikhamsai,12.459593,248.337510,357.0,1.0,5.934555,2023,1,False,Central,3,330700,2526.05,22,8080.44,258574.08
1,2023-Q1,Champasak,5.916164,182.263444,18.0,2.0,1.050758,2023,1,False,South,1,781200,2526.05,51,8080.44,258574.08
2,2023-Q1,Vientiane Capital,12.079012,177.178575,388.0,1.0,6.570471,2023,1,False,Central,1,1009300,2526.05,257,8080.44,258574.08
3,2023-Q1,Xiangkhouang,9.993296,83.874250,4.0,1.0,0.670157,2023,1,False,North,3,274400,2526.05,19,8080.44,258574.08
4,2023-Q2,Bolikhamsai,10.555417,155.074625,926.0,1.0,6.117149,2023,2,False,Central,3,330700,2526.05,22,8080.44,258574.08


In [9]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../../data/exports/ndt7_mobile_laos_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 45 rows -> ../../../data/exports/ndt7_mobile_laos_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Bolikhamsai,2023-Q1,2023,1,12.459593,5.934555,248.337510,357.0,1.0,False,Central,3,330700,2526.05,22,8080.44,258574.08
1,Champasak,2023-Q1,2023,1,5.916164,1.050758,182.263444,18.0,2.0,False,South,1,781200,2526.05,51,8080.44,258574.08
2,Vientiane Capital,2023-Q1,2023,1,12.079012,6.570471,177.178575,388.0,1.0,False,Central,1,1009300,2526.05,257,8080.44,258574.08


## Summary

- Input: Laos NDT7 raw test records, already province-joined + ISP-classified
- Output: province x quarter aggregates for Broadband and Mobile separately, tile-binned at
  Ookla's zoom-16 resolution, same `is_reliable` threshold as every Ookla country notebook and
  the other NDT7 "tigger" prep notebooks
- Engine: DuckDB (was: manual pyarrow-batch-streaming loop in pandas) — verified to reproduce
  the prior pandas-based export exactly (float-precision-only differences)